# Golden Truth Generation — Personalized Learning Insights

**Research:** An End-to-End Study on Prompt Engineering Techniques in Generative AI  
for Automated Course Understanding, Skill Extraction, and Personalized Learning Insights

---

This notebook generates **personalized learning insight golden truth records** for the Coursera dataset  
using Google Gemini as a proxy annotator.

Each gold insight record includes:
- **target_learner** — the ideal learner profile (background, role, experience level)
- **prerequisite_skills** — skills or knowledge expected before enrolling
- **skills_gained** — concrete skills a learner acquires upon completion
- **learning_outcomes** — 2–3 specific, measurable outcomes
- **career_relevance** — how the course maps to career paths or job roles
- **next_steps** — recommended follow-on courses or learning directions

**Sections**
- Cell 1: Install dependencies
- Cell 2: Generate golden truth insights and export to CSV

**Disclosure note for methodology:**  
Golden truth insights are generated by Google Gemini (a model not used in any  
experiment being evaluated), serving as a silver-standard proxy for human annotation.

---

**Setup**
- Step 1: Run Cell 1 to install dependencies. Restart runtime if prompted.
- Step 2: Add your Gemini API key as a Colab Secret named `GEMINI_API_KEY`.
- Step 3: Run Cell 2.

In [ ]:
# =============================================================================
# CELL 1 — INSTALL DEPENDENCIES
# =============================================================================

import subprocess
import sys

PACKAGES = [
    ("google-generativeai>=0.5.0",),
    ("pandas>=2.0.0",),
]


def pip_install(*packages: str) -> None:
    """Install one or more pip packages quietly via subprocess."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )


for pkg_group in PACKAGES:
    pip_install(*pkg_group)

print("All packages installed successfully.")
print("If Colab prompts a runtime restart, click Restart.")
print("After restarting, run Cell 2 directly — do NOT re-run Cell 1.")

All packages installed successfully.
If Colab prompts a runtime restart, click Restart.
After restarting, run Cell 2 directly — do NOT re-run Cell 1.


In [ ]:
# =============================================================================
# CELL 2 — PERSONALIZED LEARNING INSIGHT GOLDEN TRUTH GENERATION
#
# Sections
# --------
#   A.  Imports
#   B.  Configuration
#   C.  Dataset load
#   D.  API key setup  (supports up to 3 keys with automatic fallback)
#   E.  Prompt template
#   F.  Gemini caller
#   G.  Response parser
#   H.  Generation loop
#   I.  Quality checks
#   J.  Save & download
# =============================================================================


# =============================================================================
# A.  IMPORTS
# =============================================================================

import getpass
import os
import random
import re
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import google.generativeai as genai
import pandas as pd
from google.colab import drive

try:
    from google.colab import files
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

SEPARATOR_SM: int = 40
SEPARATOR_MD: int = 55
SEPARATOR_LG: int = 65


# =============================================================================
# B.  CONFIGURATION
# =============================================================================

# Gemini model to use as proxy annotator.
# Must NOT be any model used in the main experiment.
GEMINI_MODEL: str = "gemini-2.5-flash"

# Generation parameters
TEMPERATURE: float = 0.2   # Low temperature for consistent, factual output
TOP_P: float = 0.9
# 2048 is the minimum safe budget for 6 structured fields on the free tier.
# 1024 causes truncation after the first field — all subsequent fields come
# back empty and the record is flagged 'incomplete'. gemini-2.5-flash allows
# up to 8192 output tokens even on the free tier, so 2048 has no extra cost.
MAX_OUTPUT_TOKENS: int = 2048

# Inter-call delay to respect API rate limits (seconds)
DELAY_MIN: float = 8.0
DELAY_MAX: float = 12.0

# Retry settings for API calls
MAX_RETRIES: int = 3
RETRY_BACKOFF_MIN: float = 10.0
RETRY_BACKOFF_MAX: float = 20.0

# Max retries for insight generation if any required field is missing
MAX_GENERATION_RETRIES: int = 2  # Including initial attempt, so 2 retries = 3 total attempts

# 503 retry settings — separate budget so transient server overload does
# not consume the main MAX_RETRIES allowance meant for real errors.
MAX_503_RETRIES: int = 5          # silent retries before giving up on 503
BACKOFF_503_MIN: float = 15.0     # longer backoff: 503 = server congestion
BACKOFF_503_MAX: float = 30.0

# Dataset
N_ROWS: int = 25

# Names of the three Colab Secrets — add each key under the matching name
GEMINI_SECRET_NAMES: List[str] = [
    "GEMINI_API_KEY",    # Key 1 (primary)
    "GEMINI_API_KEY_2",  # Key 2 (first fallback)
    "GEMINI_API_KEY_3",  # Key 3 (second fallback)
]

# The six structured fields the model must populate.
# Used for completeness validation in quality checks.
REQUIRED_INSIGHT_FIELDS: List[str] = [
    "target_learner",
    "prerequisite_skills",
    "skills_gained",
    "learning_outcomes",
    "career_relevance",
    "next_steps",
]

# Minimum word count per insight field to be considered non-trivial
MIN_FIELD_WORD_COUNT: int = 5


def print_config() -> None:
    """Print the current generation configuration."""
    total_calls = N_ROWS * (MAX_GENERATION_RETRIES + 1)  # Max possible calls
    avg_delay_min = total_calls * (DELAY_MIN + DELAY_MAX) / 2 / 60
    print("Golden Truth Generation — Personalized Learning Insights — Configuration")
    print("-" * SEPARATOR_LG)
    print(f"  Gemini model       : {GEMINI_MODEL}")
    print(f"  Temperature        : {TEMPERATURE}")
    print(f"  top_p              : {TOP_P}")
    print(f"  Max output tokens  : {MAX_OUTPUT_TOKENS}")
    print(f"  Courses            : {N_ROWS}")
    print(f"  Max API retries    : {MAX_RETRIES}")
    print(f"  Max gen retries    : {MAX_GENERATION_RETRIES}")
    print(f"  Est. total calls   : {total_calls} (max possible)")
    print(f"  Est. time (max)    : ~{avg_delay_min:.1f} min")
    print(f"  Required fields    : {len(REQUIRED_INSIGHT_FIELDS)}")
    print(f"  Min words/field    : {MIN_FIELD_WORD_COUNT}")
    print(f"  API keys available : {len(GEMINI_SECRET_NAMES)}")


print_config()


# =============================================================================
# C.  DATASET LOAD
# =============================================================================

COLUMN_RENAME_MAP: Dict[str, str] = {
    "title":        "course_title",
    "Organization": "course_organization",
    "Level":        "course_difficulty",
    "Description":  "description",
    "enrolled":     "course_students_enrolled",
    "rating":       "course_rating",
}

TEXT_COLUMNS: List[str] = [
    "course_title",
    "course_organization",
    "course_difficulty",
    "description",
]


def load_dataset(n_rows: int = N_ROWS) -> pd.DataFrame:
    """Load and clean the Coursera CSV dataset from Google Drive.

    Args:
        n_rows: Maximum number of rows to load.

    Returns:
        Cleaned DataFrame with standardised columns.

    Raises:
        FileNotFoundError: If the dataset CSV is not found at the expected path.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("Dataset Load")
    print("-" * SEPARATOR_SM)

    drive.mount("/content/drive")

    dataset_path = "/content/drive/MyDrive/research/data/coursera_clean_sample.csv"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Dataset not found at '{dataset_path}'.\n"
            "Please ensure the file exists in your Google Drive."
        )

    print(f"  Found dataset at: {dataset_path}")

    df = pd.read_csv(dataset_path, encoding="latin-1")
    df = df.rename(columns=COLUMN_RENAME_MAP)
    df = df.dropna(subset=["course_title"]).head(n_rows).reset_index(drop=True)

    print(f"  Loaded {len(df)} courses successfully.")
    return df


DATASET_DF: pd.DataFrame = load_dataset(N_ROWS)


# =============================================================================
# D.  API KEY SETUP  (up to 3 keys, automatic quota-based fallback)
# =============================================================================

def load_api_key(secret_name: str, key_number: int) -> str:
    """Load a single Gemini API key from Colab Secrets or manual input.

    Args:
        secret_name: The Colab Secret key name (e.g. "GEMINI_API_KEY").
        key_number:  Human-readable key number (1, 2, or 3) for prompts.

    Returns:
        API key string, or empty string if the key was not provided.
    """
    api_key = ""

    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get(secret_name).strip()
            print(f"  Key {key_number} loaded from Colab Secret '{secret_name}'.")
        except Exception:
            pass  # Key not set — will try manual entry for key 1 only

    if not api_key and key_number == 1:
        # Always prompt for the primary key if it is missing
        api_key = getpass.getpass(
            f"  Enter your primary Gemini API key ('{secret_name}'): "
        ).strip()

    if api_key:
        print(f"  Key {key_number} : AIza...{api_key[-4:]}")
    else:
        print(f"  Key {key_number} : not set (secret '{secret_name}' not found — skipped).")

    return api_key


def load_all_api_keys(secret_names: List[str]) -> List[str]:
    """Load all configured API keys, returning only non-empty ones.

    Args:
        secret_names: Ordered list of Colab Secret names to try.

    Returns:
        List of non-empty API key strings, in priority order.

    Raises:
        ValueError: If no valid key is found.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("API Key Setup")
    print("-" * SEPARATOR_SM)

    keys: List[str] = []
    for i, name in enumerate(secret_names, start=1):
        key = load_api_key(name, i)
        if key:
            keys.append(key)

    if not keys:
        raise ValueError(
            "No Gemini API key provided. "
            f"Add at least one key as a Colab Secret named '{secret_names[0]}' "
            "or enter it when prompted."
        )

    print(f"\n  {len(keys)} key(s) available for automatic fallback.")
    return keys


def build_gemini_client(api_key: str) -> genai.GenerativeModel:
    """Configure the genai library with the given key and return a client.

    Args:
        api_key: Gemini API key string.

    Returns:
        Configured GenerativeModel instance.
    """
    genai.configure(api_key=api_key)
    return genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        generation_config=genai.GenerationConfig(
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        ),
        # System instruction enforces plain-label output at the model level.
        # Prevents **bold** and ### heading decorators on field labels,
        # which break the regex parser in parse_insight_response().
        system_instruction=(
            "You are a structured data generator. "
            "Always respond using plain uppercase labels followed by a colon, "
            "e.g. TARGET_LEARNER: value. "
            "Never use markdown bold (**), asterisks, or heading hashes (#). "
            "Never add preamble or closing text outside the six required fields."
        ),
    )


ALL_API_KEYS: List[str] = load_all_api_keys(GEMINI_SECRET_NAMES)

# Active key index; incremented on each quota error
_active_key_index: int = 0
GEMINI_CLIENT = build_gemini_client(ALL_API_KEYS[_active_key_index])
print(f"  Gemini client ready : {GEMINI_MODEL}  (using key 1 of {len(ALL_API_KEYS)})")


def rotate_api_key() -> bool:
    """Switch to the next available API key and rebuild the Gemini client.

    Returns:
        True if rotation succeeded, False if all keys are exhausted.
    """
    global _active_key_index, GEMINI_CLIENT

    next_index = _active_key_index + 1
    if next_index >= len(ALL_API_KEYS):
        return False  # No more keys

    _active_key_index = next_index
    GEMINI_CLIENT = build_gemini_client(ALL_API_KEYS[_active_key_index])
    print(
        f"\n  *** Rotated to API key {_active_key_index + 1} of {len(ALL_API_KEYS)} ***\n"
    )
    return True


# =============================================================================
# E.  PROMPT TEMPLATE
# =============================================================================

def build_personalized_insight_prompt(
    title: str,
    organization: str,
    difficulty: str,
    description: str,
) -> str:
    """Build the personalized learning insight prompt for a single course.

    The prompt instructs Gemini to act as a learning advisor and produce
    six structured insight fields. All six fields must be populated with
    original, specific content — not generic placeholders.

    The response must follow the exact label format shown so the parser
    can extract each field reliably.

    Args:
        title:        Course title string.
        organization: Course provider string.
        difficulty:   Difficulty level string.
        description:  Cleaned course description string.

    Returns:
        Fully rendered prompt string.
    """
    return f"""You are an expert learning advisor and curriculum analyst.

Task: Generate a structured personalized learning insight record for the course below.

Requirements:
- Be specific to this course — avoid generic statements.
- Use your own wording; do not copy sentences from the description.
- Every field must be populated with substantive content.
- Keep each field concise (1–3 sentences or a short list).

Input:
Course Title : {title}
Organization : {organization}
Difficulty   : {difficulty}
Description  : {description}


TARGET_LEARNER: <who this course is best suited for — role, background, experience level>
PREREQUISITE_SKILLS: <knowledge or skills a learner should have before enrolling>
SKILLS_GAINED: <concrete, transferable skills acquired upon completion>
LEARNING_OUTCOMES: <2–3 specific, measurable things a learner will be able to do>
CAREER_RELEVANCE: <how this course maps to job roles, industries, or career advancement>
NEXT_STEPS: <recommended follow-on courses, certifications, or learning directions>"""


# =============================================================================
# F.  GEMINI CALLER
# =============================================================================

# Phrases that indicate a quota or rate-limit error
QUOTA_ERROR_PHRASES: Tuple[str, ...] = (
    "quota",
    "rate limit",
    "resource exhausted",
    "429",
)


class QuotaExceeded(Exception):
    """Raised when ALL available Gemini API keys have hit their quota.

    At this point no rotation is possible and the caller saves partial results.
    """


def is_quota_error(exc: Exception) -> bool:
    """Return True if the exception signals an API quota error.

    Args:
        exc: Exception raised by the Gemini client.

    Returns:
        True if any quota-related phrase is found in the error message.
    """
    return any(phrase in str(exc).lower() for phrase in QUOTA_ERROR_PHRASES)


def call_gemini(
    prompt: str,
    retries: int = MAX_RETRIES,
) -> Tuple[str, float]:
    """Call the Gemini API and return (response_text, latency_seconds).

    Applies a random inter-call delay before each attempt.
    503 Service Unavailable errors use a separate retry budget (MAX_503_RETRIES)
    with longer back-off (BACKOFF_503_MIN/MAX) so transient server congestion
    does not consume the main `retries` allowance reserved for real errors.
    On quota errors, automatically rotates to the next available API key
    and retries immediately. Raises QuotaExceeded only when all keys are
    exhausted. Retries up to `retries` times with random back-off on
    other transient errors.

    Args:
        prompt:  Fully rendered prompt string.
        retries: Number of retry attempts on transient errors.

    Returns:
        Tuple of (response_text, latency_in_seconds).

    Raises:
        QuotaExceeded: If all API keys have hit their quota limit.
    """
    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"    Waiting {delay:.1f}s...", end=" ", flush=True)
    time.sleep(delay)

    for attempt in range(1, retries + 1):
        t_start = time.perf_counter()
        try:
            response = GEMINI_CLIENT.generate_content(prompt)
            latency = round(time.perf_counter() - t_start, 3)
            text = response.text.strip() if response.text else ""
            print(f"OK ({latency}s)")
            return text, latency

        except Exception as exc:
            latency = round(time.perf_counter() - t_start, 3)

            if "503" in str(exc) or "service unavailable" in str(exc).lower():
                # 503 = Gemini server overload — use a dedicated retry budget
                # so these silent retries do not consume MAX_RETRIES slots.
                _503_attempt = 0
                while _503_attempt < MAX_503_RETRIES:
                    _503_attempt += 1
                    backoff_503 = random.uniform(BACKOFF_503_MIN, BACKOFF_503_MAX)
                    print(
                        f"\n    503 overload (retry {_503_attempt}/{MAX_503_RETRIES}) "
                        f"— waiting {backoff_503:.0f}s...",
                        end=" ", flush=True,
                    )
                    time.sleep(backoff_503)
                    t_retry = time.perf_counter()
                    try:
                        response = GEMINI_CLIENT.generate_content(prompt)
                        latency = round(time.perf_counter() - t_retry, 3)
                        text = response.text.strip() if response.text else ""
                        print(f"OK ({latency}s)")
                        return text, latency
                    except Exception as retry_exc:
                        if "503" not in str(retry_exc) and "service unavailable" not in str(retry_exc).lower():
                            # Different error — break out and let the outer handler deal with it
                            exc = retry_exc
                            break
                        print(f"still 503...", end=" ", flush=True)
                else:
                    print(f"\n    503 persisted after {MAX_503_RETRIES} retries. Recording empty insight.")
                    return "", round(time.perf_counter() - t_start, 3)
                # Fall through to quota / generic error handling below

            if is_quota_error(exc):
                print(f"\n    Quota limit hit on key {_active_key_index + 1}.")
                rotated = rotate_api_key()
                if not rotated:
                    raise QuotaExceeded(
                        f"All {len(ALL_API_KEYS)} API key(s) have hit their quota. "
                        "Original error: " + str(exc)
                    ) from exc
                print(f"    Retrying with key {_active_key_index + 1}...")
                delay2 = random.uniform(DELAY_MIN, DELAY_MAX)
                print(f"    Waiting {delay2:.1f}s...", end=" ", flush=True)
                time.sleep(delay2)
                attempt = 0  # restart the for-loop logically
                continue

            print(f"\n    Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                print(f"    Retrying in {backoff:.1f}s...")
                time.sleep(backoff)
            else:
                print("    All retries exhausted. Recording empty insight.")
                return "", latency


# =============================================================================
# G.  RESPONSE PARSER
# =============================================================================

# Maps the label token in the response to the output column name
FIELD_LABEL_MAP: Dict[str, str] = {
    "TARGET_LEARNER":      "target_learner",
    "PREREQUISITE_SKILLS": "prerequisite_skills",
    "SKILLS_GAINED":       "skills_gained",
    "LEARNING_OUTCOMES":   "learning_outcomes",
    "CAREER_RELEVANCE":    "career_relevance",
    "NEXT_STEPS":          "next_steps",
}


def _normalise_response(raw_text: str) -> str:
    """Strip markdown decorators that Gemini adds around field labels.

    Gemini frequently ignores plain-label instructions and returns responses
    with bold markers or heading hashes, e.g.:
        **TARGET_LEARNER**: ...   ->  TARGET_LEARNER: ...
        ### TARGET_LEARNER: ...  ->  TARGET_LEARNER: ...
        **TARGET_LEARNER:**      ->  TARGET_LEARNER:

    This normaliser runs before the splitter so the regex always sees bare
    UPPER_CASE labels regardless of how Gemini formatted them.

    Args:
        raw_text: Raw string returned by the Gemini API.

    Returns:
        Normalised string with markdown decorators removed from labels.
    """
    # Remove leading markdown headings (###, ##, #) before a label
    text = re.sub(r'^#{1,6}\s*', '', raw_text, flags=re.MULTILINE)
    # Remove bold/italic markers wrapping labels: **LABEL**: or **LABEL**:
    text = re.sub(r'\*{1,2}([A-Z_]+)\*{1,2}\s*:', r'\1:', text)
    # Remove any remaining stray asterisks immediately before a known label
    text = re.sub(r'^\*+\s*([A-Z_]+)\s*:', r'\1:', text, flags=re.MULTILINE)
    return text


def _clean_field_value(text: str) -> str:
    """Normalise a parsed field value by removing common Gemini formatting artefacts.

    Handles all observed patterns across fields, in order:
      1. Leading '=' or '==' signs  (e.g. '= Outcome text')
      2. Numbered list markers      (e.g. '1. item', '1) item', '1 - item')
      3. Bullet/dash markers        (e.g. '- item', '* item', '• item')
      4. Stray leading punctuation  (e.g. ':' or '|' left after stripping)

    Each line in the value is cleaned independently so multi-line fields
    (e.g. LEARNING_OUTCOMES with three numbered lines) are fully normalised.

    Args:
        text: Raw extracted field value string.

    Returns:
        Cleaned string with artefacts removed and whitespace normalised.
    """
    if not text:
        return text

    cleaned_lines: List[str] = []
    for line in text.splitlines():
        line = line.strip()
        # 1. Leading '=' or '==' (e.g. '= Outcome one')
        line = re.sub(r'^=+\s*', '', line)
        # 2. Numbered list markers: '1.', '1)', '1-', '1 -'
        line = re.sub(r'^\d+[.)\-]\s*|^\d+\s+-\s*', '', line)
        # 3. Bullet / dash / asterisk markers
        line = re.sub(r'^[-*•–]\s*', '', line)
        # 4. Stray leading colon or pipe left after stripping
        line = re.sub(r'^[:|]\s*', '', line)
        if line:
            cleaned_lines.append(line)

    return ' '.join(cleaned_lines).strip()


def parse_insight_response(raw_text: str) -> Dict[str, str]:
    """Extract the six structured insight fields from a Gemini response.

    Normalises markdown decorators first, then splits on bare LABEL: tokens.
    Handles all observed Gemini output styles:
        TARGET_LEARNER: value          (plain — ideal)
        **TARGET_LEARNER**: value      (bold label)
        **TARGET_LEARNER:** value      (bold label + colon inside)
        ### TARGET_LEARNER: value      (heading prefix)

    Field values are passed through _clean_field_value() which strips:
        '= Outcome text'               (leading equals sign)
        '1. outcome' / '1) outcome'    (numbered list markers)
        '- item' / '* item'            (bullet markers)

    Multi-line values are captured until the next label or end of string.
    Missing fields are returned as empty strings so downstream status
    checks can flag them.

    Args:
        raw_text: Raw string returned by the Gemini API.

    Returns:
        Dictionary mapping output column names to their extracted values.
    """
    parsed: Dict[str, str] = {col: "" for col in FIELD_LABEL_MAP.values()}

    if not raw_text:
        return parsed

    normalised = _normalise_response(raw_text)

    # Build a regex that matches any bare label at the start of a line
    label_pattern = "|".join(re.escape(lbl) for lbl in FIELD_LABEL_MAP)
    splitter = re.compile(
        rf"^({label_pattern})\s*:\s*",
        re.MULTILINE,
    )

    parts = splitter.split(normalised)
    # parts alternates: [preamble, label, value, label, value, ...]
    # preamble at index 0 is discarded (e.g. "Here is the record:
    it = iter(parts[1:])  # skip preamble
    for label, value in zip(it, it):
        col_name = FIELD_LABEL_MAP.get(label.strip())
        if col_name:
            parsed[col_name] = _clean_field_value(value.strip())

    return parsed


def compute_insight_status(
    parsed: Dict[str, str],
    raw_text: str,
) -> str:
    """Determine the status of a parsed insight record.

    Status codes:
        "ok"            — all six fields present and meet minimum word count.
        "incomplete"    — one or more required fields missing or too short.
        "empty"         — the raw API response was blank.

    Args:
        parsed:   Dictionary of parsed field values.
        raw_text: Original raw API response string.

    Returns:
        Status string.
    """
    if not raw_text:
        return "empty"

    for field in REQUIRED_INSIGHT_FIELDS:
        value = parsed.get(field, "")
        if not value or len(value.split()) < MIN_FIELD_WORD_COUNT:
            return "incomplete"

    return "ok"


# =============================================================================
# H.  GENERATION LOOP
# =============================================================================

def build_gold_record(
    idx: int,
    row: pd.Series,
    parsed: Dict[str, str],
    raw_text: str,
    latency: float,
    status: str,
) -> Dict:
    """Assemble a single gold insight record dict.

    Args:
        idx:      Row index in the dataset.
        row:      Dataset row as a Series.
        parsed:   Dictionary of parsed insight fields.
        raw_text: Raw Gemini response string.
        latency:  API call latency in seconds.
        status:   "ok", "incomplete", or "empty".

    Returns:
        Dictionary with all gold record fields.
    """
    total_words = sum(
        len(parsed.get(f, "").split()) for f in REQUIRED_INSIGHT_FIELDS
    )

    return {
        "course_idx":           idx,
        "course_title":         str(row.get("course_title", "Unknown")),
        "course_organization":  str(row.get("course_organization", "N/A")),
        "course_difficulty":    str(row.get("course_difficulty", "N/A")),
        "description":          str(row.get("description", "")),
        # Six structured insight fields
        "target_learner":       parsed.get("target_learner", ""),
        "prerequisite_skills":  parsed.get("prerequisite_skills", ""),
        "skills_gained":        parsed.get("skills_gained", ""),
        "learning_outcomes":    parsed.get("learning_outcomes", ""),
        "career_relevance":     parsed.get("career_relevance", ""),
        "next_steps":           parsed.get("next_steps", ""),
        # Metadata
        "raw_response":         raw_text,
        "total_insight_words":  total_words,
        "latency_seconds":      latency,
        "status":               status,
        "gemini_model":         GEMINI_MODEL,
        "api_key_index":        _active_key_index + 1,
        "temperature":          TEMPERATURE,
        "timestamp":            datetime.now(timezone.utc).isoformat(),
    }


gold_records: List[Dict] = []
quota_hit: bool = False
generation_ts: str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\n" + "=" * SEPARATOR_LG)
print("GENERATING PERSONALIZED LEARNING INSIGHT GOLDEN TRUTH RECORDS")
print("=" * SEPARATOR_LG)

try:
    for idx, row in DATASET_DF.iterrows():
        title        = str(row.get("course_title", "Unknown"))
        organization = str(row.get("course_organization", "N/A"))
        difficulty   = str(row.get("course_difficulty", "N/A"))
        description  = str(row.get("description", ""))

        print(f"  [{idx:02d}] {title[:60]}...")

        prompt = build_personalized_insight_prompt(
            title, organization, difficulty, description
        )

        current_parsed:   Dict[str, str] = {col: "" for col in FIELD_LABEL_MAP.values()}
        current_raw:      str   = ""
        current_latency:  float = 0.0
        current_status:   str   = "empty"

        for attempt in range(MAX_GENERATION_RETRIES + 1):
            if attempt > 0:
                print(
                    f"    Retrying generation (attempt {attempt + 1}/{MAX_GENERATION_RETRIES + 1}) "
                    "for 'incomplete' insight..."
                )
                time.sleep(random.uniform(2.0, 5.0))

            raw_text, latency = call_gemini(prompt)
            parsed            = parse_insight_response(raw_text)
            status            = compute_insight_status(parsed, raw_text)

            current_parsed  = parsed
            current_raw     = raw_text
            current_latency = latency
            current_status  = status

            # Log which fields are missing on incomplete records
            if status == "incomplete":
                missing = [
                    f for f in REQUIRED_INSIGHT_FIELDS
                    if len(parsed.get(f, "").split()) < MIN_FIELD_WORD_COUNT
                ]
                print(f"    Incomplete — missing/short fields: {missing}")
                # Show a snippet of the raw response to diagnose truncation vs
                # label-format mismatch. If the text ends mid-sentence the
                # cause is token budget; if labels look like '**LABEL:**' the
                # cause is markdown decoration (fix: increase MAX_OUTPUT_TOKENS
                # or add 'no markdown' to the prompt).
                snippet = raw_text[:400] if raw_text else "<empty>"
                print(f"    --- raw response snippet (first 400 chars) ---")
                print(f"    {snippet}")
                print(f"    --- end snippet (total chars: {len(raw_text)}) ---")

            if status == "ok":
                break  # All fields present; exit retry loop

        gold_records.append(
            build_gold_record(
                idx, row, current_parsed, current_raw, current_latency, current_status
            )
        )

except QuotaExceeded as exc:
    quota_hit = True
    print("\n" + "!" * SEPARATOR_LG)
    print("*** ALL API KEYS QUOTA EXCEEDED — generation stopped here ***")
    print(f"  Records collected : {len(gold_records)}")
    print(f"  Error             : {exc}")
    print("!" * SEPARATOR_LG)

GOLD_DF: pd.DataFrame = pd.DataFrame(gold_records)
generation_status = "STOPPED EARLY (all keys quota exhausted)" if quota_hit else "COMPLETE"
print(f"\nGeneration {generation_status}. Total records: {len(GOLD_DF)}")


# =============================================================================
# I.  QUALITY CHECKS
# =============================================================================

def run_quality_checks(gold_df: pd.DataFrame) -> None:
    """Print a quality report on the generated gold insight records.

    Checks performed:
    - Status distribution (ok / incomplete / empty)
    - Per-field completeness: word count stats for each of the six fields
    - Total insight word count statistics
    - Verbatim copying check: flags any field that shares a 10-word
      contiguous span with its source description
    - API key usage breakdown
    - Latency summary

    Args:
        gold_df: DataFrame produced by the generation loop.
    """
    if gold_df.empty:
        print("No records to check.")
        return

    print("\n" + "=" * SEPARATOR_LG)
    print("QUALITY REPORT")
    print("=" * SEPARATOR_LG)

    # Status distribution
    print("\n  Status distribution")
    print("  " + "-" * SEPARATOR_SM)
    for status, count in gold_df["status"].value_counts().items():
        pct = count / len(gold_df) * 100
        print(f"    {status:<15} {count:>3}  ({pct:.1f}%)")

    # Per-field word count statistics (ok records only)
    ok_df = gold_df[gold_df["status"] == "ok"].copy()
    print("\n  Per-field word counts (status=ok records)")
    print("  " + "-" * SEPARATOR_MD)
    if ok_df.empty:
        print("    No ok records available for field statistics.")
    else:
        field_col_width = max(len(f) for f in REQUIRED_INSIGHT_FIELDS) + 2
        header = (
            f"    {'Field':<{field_col_width}}"
            f"  {'Min':>5}  {'Max':>5}  {'Mean':>6}  {'Median':>7}"
        )
        print(header)
        print("    " + "-" * (field_col_width + 28))
        for field in REQUIRED_INSIGHT_FIELDS:
            wc = ok_df[field].apply(lambda x: len(str(x).split()) if x else 0)
            print(
                f"    {field:<{field_col_width}}"
                f"  {wc.min():>5}  {wc.max():>5}  {wc.mean():>6.1f}  {wc.median():>7.1f}"
            )

    # Total insight words
    if not ok_df.empty:
        tw = ok_df["total_insight_words"]
        print("\n  Total insight word count (status=ok)")
        print("  " + "-" * SEPARATOR_SM)
        print(f"    Min    : {tw.min()}")
        print(f"    Max    : {tw.max()}")
        print(f"    Mean   : {tw.mean():.1f}")
        print(f"    Median : {tw.median():.1f}")

    # Incomplete record detail
    incomplete_df = gold_df[gold_df["status"] == "incomplete"]
    if not incomplete_df.empty:
        print("\n  Incomplete records — missing / short fields")
        print("  " + "-" * SEPARATOR_MD)
        for _, row in incomplete_df.iterrows():
            missing = [
                f for f in REQUIRED_INSIGHT_FIELDS
                if len(str(row.get(f, "")).split()) < MIN_FIELD_WORD_COUNT
            ]
            print(
                f"    [{int(row['course_idx']):02d}] "
                f"{str(row['course_title'])[:45]}...  →  {missing}"
            )

    # Verbatim copying check (10-word ngram overlap with description)
    VERBATIM_NGRAM_SIZE: int = 10

    def get_word_ngrams(text: str, n: int) -> set:
        words = text.lower().split()
        if len(words) < n:
            return set()
        return {tuple(words[i: i + n]) for i in range(len(words) - n + 1)}

    copying_flags: List[Dict] = []
    for _, row in gold_df.iterrows():
        description = str(row.get("description", "")) or ""
        desc_ngrams = get_word_ngrams(description, VERBATIM_NGRAM_SIZE)
        field_overlaps: Dict[str, int] = {}
        for field in REQUIRED_INSIGHT_FIELDS:
            field_text = str(row.get(field, "")) or ""
            field_ngrams = get_word_ngrams(field_text, VERBATIM_NGRAM_SIZE)
            overlap = field_ngrams & desc_ngrams
            if overlap:
                field_overlaps[field] = len(overlap)
        if field_overlaps:
            copying_flags.append({
                "course_idx":    row["course_idx"],
                "course_title":  row["course_title"],
                "field_overlaps": field_overlaps,
            })

    print(f"\n  Verbatim copying check ({VERBATIM_NGRAM_SIZE}-word span overlap with description)")
    print("  " + "-" * SEPARATOR_MD)
    if copying_flags:
        print(f"    Flagged : {len(copying_flags)} records")
        print("    Review the following before using as gold truth:")
        for flag in copying_flags:
            print(
                f"      [{int(flag['course_idx']):02d}] {str(flag['course_title'])[:45]}...  "
                f"overlapping fields: {flag['field_overlaps']}"
            )
    else:
        print(f"    No verbatim copying detected across all {len(gold_df)} records.")

    # API key usage breakdown
    if "api_key_index" in gold_df.columns:
        print("\n  API key usage")
        print("  " + "-" * SEPARATOR_SM)
        for key_idx, count in gold_df["api_key_index"].value_counts().sort_index().items():
            print(f"    Key {key_idx} : {count} record(s)")

    # Latency summary
    lat = gold_df["latency_seconds"]
    print("\n  Latency (seconds)")
    print("  " + "-" * SEPARATOR_SM)
    print(f"    Min    : {lat.min():.3f}s")
    print(f"    Max    : {lat.max():.3f}s")
    print(f"    Mean   : {lat.mean():.3f}s")
    print(f"    Total  : {lat.sum():.1f}s")

    print("\n" + "=" * SEPARATOR_LG)


run_quality_checks(GOLD_DF)


# =============================================================================
# J.  SAVE & DOWNLOAD
# =============================================================================

GOLD_FILENAME:    str = f"gold_learning_insights_{generation_ts}.csv"
FLAGGED_FILENAME: str = f"gold_learning_insights_flagged_{generation_ts}.csv"


def save_gold_insights(
    gold_df: pd.DataFrame,
    in_colab: bool,
) -> None:
    """Save the gold insight DataFrame to CSV and optionally trigger downloads.

    Saves two files:
    - Full output: all records including failures.
    - Flagged output: only records with status != 'ok', for manual review.

    Args:
        gold_df:  DataFrame produced by the generation loop.
        in_colab: True if running inside Google Colab.
    """
    gold_df.to_csv(GOLD_FILENAME, index=False)
    print(f"\nSaved: {GOLD_FILENAME}  ({len(gold_df)} rows)")

    flagged_df = gold_df[gold_df["status"] != "ok"]
    if not flagged_df.empty:
        flagged_df.to_csv(FLAGGED_FILENAME, index=False)
        print(
            f"Saved: {FLAGGED_FILENAME}  ({len(flagged_df)} rows — manual review needed)"
        )
    else:
        print("No flagged records — all insight records passed status checks.")

    if in_colab:
        print("\nDownloading files...")
        files.download(GOLD_FILENAME)
        if not flagged_df.empty:
            files.download(FLAGGED_FILENAME)
        print("Downloads initiated.")
    else:
        print("\nFiles saved locally.")

    print("\nDone.")


save_gold_insights(GOLD_DF, IN_COLAB)

Golden Truth Generation — Personalized Learning Insights — Configuration
-----------------------------------------------------------------
  Gemini model       : gemini-2.5-flash
  Temperature        : 0.2
  top_p              : 0.9
  Max output tokens  : 2048
  Courses            : 25
  Max API retries    : 3
  Max gen retries    : 2
  Est. total calls   : 75 (max possible)
  Est. time (max)    : ~12.5 min
  Required fields    : 6
  Min words/field    : 5
  API keys available : 3

----------------------------------------
Dataset Load
----------------------------------------
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Found dataset at: /content/drive/MyDrive/research/data/coursera_clean_sample.csv
  Loaded 25 courses successfully.

----------------------------------------
API Key Setup
----------------------------------------
  Key 1 loaded from Colab Secret 'GEMINI_API_KEY'.
  Key 1 : AIza...HCJM
  

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2078.28ms


OK (12.746s)
  [11] financial reporting specialization...
    Waiting 11.1s... OK (12.577s)
  [12] modernizing data lakes and data warehouses with google cloud...
    Waiting 11.8s... OK (8.474s)
  [13] introduction to data analytics for accounting professionals...
    Waiting 8.1s... OK (9.74s)
  [14] statistics in psychological research...
    Waiting 10.6s... OK (8.745s)
  [15] plastic electronics...
    Waiting 8.7s... OK (9.39s)
  [16] learn english beginning grammar specialization...
    Waiting 8.7s... OK (6.843s)
  [17] digital creativity...
    Waiting 8.8s... OK (6.82s)
  [18] crisis communications...
    Waiting 11.0s... OK (6.671s)
  [19] mv substation an industrial approach part a...
    Waiting 9.5s... OK (7.021s)
  [20] designing effective science communication specialization...
    Waiting 9.0s... 


    Quota limit hit on key 1.

  *** Rotated to API key 2 of 3 ***

    Retrying with key 2...
    Waiting 11.5s... OK (7.782s)
  [21] teach english now lesson design and assessment...
    Waiting 11.9s... OK (7.989s)
  [22] elastic google cloud infrastructure scaling and automation...
    Waiting 11.7s... OK (10.727s)
  [23] genai for devops practitioners...
    Waiting 11.9s... OK (6.297s)
  [24] the science of the solar system...
    Waiting 9.8s... OK (9.256s)

Generation COMPLETE. Total records: 25

QUALITY REPORT

  Status distribution
  ----------------------------------------
    ok               25  (100.0%)

  Per-field word counts (status=ok records)
  -------------------------------------------------------
    Field                    Min    Max    Mean   Median
    -------------------------------------------------
    target_learner            30     50    41.1     41.0
    prerequisite_skills       23     41    32.9     32.0
    skills_gained             19     60    38.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads initiated.

Done.
